# Molecular hydrogen

`hydrogen/molecular/basic` resolves H₂ in the vibrational levels of its ground state; `hydrogen/molecular/ions` adds H⁻ and H₃⁺. The states, processes and limits are in [Molecular hydrogen](/OpenCR/guides/molecular-hydrogen/).

In [ ]:
import plasma_plots

import opencr

model = opencr.load_model("hydrogen/molecular/basic")
print(model)
drivers = {"H2(X,v=0)": 1e19, "H(n=1)": 1e18, "H+": 5e16, "H2+": 5e16}

## The vibrational distribution

Electron collisions pump the vibrational levels; with no V–T relaxation in the model the ladder is set by e–V excitation, its inverse and the losses to dissociation and ionization:

In [ ]:
with plasma_plots.figure(1, 1) as fig:
    for temperature in (1.0, 3.0, 10.0):
        result = model.solve_steady_state(
            opencr.Conditions(temperature, 1e17, densities=drivers)
        )
        ladder = [result.population(f"H2(X,v={v})") / 1e19 for v in range(15)]
        fig[0].semilogy(range(15), ladder, "o-", label=f"$T_e$ = {temperature:g} eV")
    fig[0].set_xlabel("v")
    fig[0].set_ylabel("$n_v / n_0$")
    fig[0].legend()

## Effective rates

The quasi-static reduction gives the effective ionization and dissociation of H₂ through all its levels, per electron and per molecule in v = 0, and the electron energy lost per molecule:

In [ ]:
from opencr.analysis.quantities import quantity

for temperature in (2.0, 5.0, 20.0):
    result = model.solve_steady_state(
        opencr.Conditions(temperature, 1e17, densities=drivers)
    )
    ionization = quantity(result, "effective_rate:H2(X,v=0)>H2+")
    atoms = quantity(result, "effective_rate:H2(X,v=0)>H(n=1)")
    cooling = quantity(result, "electron_cooling:H2(X,v=0)")
    print(
        f"Te {temperature:4.1f} eV: ionization {ionization:.2e}, atoms {atoms:.2e} m^3/s,"
        f" energy loss {cooling:.2e} eV m^3/s"
    )

## Negative ions

With the ion chemistry, H⁻ is made by dissociative attachment to vibrationally excited H₂ and destroyed by electrons, ions and atoms. A wall loss stands in for the transport of H₃⁺:

In [ ]:
ions = opencr.load_model("hydrogen/molecular/ions", wall_loss={"H3+": 1e5})
result = ions.solve_steady_state(opencr.Conditions(1.0, 1e17, densities=drivers))
print(f"H- {result.population('H-'):.2e} m^-3, H3+ {result.population('H3+'):.2e} m^-3")
for row in result.dominant_processes("H-", 4):
    print(row)